# Anomalib Exploration
## Objective
Understand anomalib structure, pipeline, some generalities and get a feeling about it.

## Tutorial

In [1]:
from anomalib.data import MVTecAD2
from anomalib.deploy import ExportType
from anomalib.engine import Engine
from anomalib.models import Patchcore
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import Callback

/home/antoine/manuf-anom-detector/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


I had to create this callback class, for I encountered many errors during the 15 mn (which turned out 2 hours, at least) tutorial.
It seems (as described below) that the internal df loads mask paths as floats (nan) instead of None, which is the expected format for lightning.

this is still very much WIP, I suspect I have a validation mask path issue, but that's a topic for tomorrow!

In [2]:
class DebugCallback(Callback):
    def on_train_start(self, trainer, pl_module):
        print("DEBUG: train start")

    def on_train_batch_start(self, trainer, pl_module, batch, batch_idx):
        print("DEBUG: train batch start", batch_idx)

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        print("DEBUG: train batch end", batch_idx)

        print(type(pl_module))
        print(type(pl_module.model))
        [name for name in dir(pl_module.model) if "embed" in name.lower()]

        if hasattr(pl_module.model, "embedding_store"):
            store = pl_module.model.embedding_store
            print("DEBUG store type:", type(store))

    def on_validation_start(self, trainer, pl_module):
        print("DEBUG: validation start")

    def on_validation_batch_start(self, trainer, pl_module, batch, batch_idx):
        print("DEBUG: validation batch start", batch_idx)


In [3]:
datamodule = MVTecAD2(
    root="./../data",
    category='vial',
    train_batch_size = 32,
    eval_batch_size=32,
)

In [4]:
model = Patchcore(
    num_neighbors=6
)

In [5]:
engine = Engine(max_epochs=1, enable_progress_bar=False, callbacks=[DebugCallback()])

In [30]:
#anomalib loads mask path as floats, I thus have to convert the path to an object and then force the None value for images without masks.
#This is needed because I encountered an error with the mask_path being a float for images without a mask (eg training images)
datamodule.prepare_data()
datamodule.setup()

In [31]:
def objectify_mask_paths(x):
    x.samples['mask_path']=x.samples['mask_path'].astype(object)
    x.samples['mask_path']=x.samples['mask_path'].where(x.samples['mask_path'].notna(), None)
    return x

In [32]:
[a for a in dir(datamodule) if "data" in a.lower()]

#datamodule.test_data = objectify_mask_paths(test_data)


datamodule.test_data=objectify_mask_paths(datamodule.test_data)
datamodule.test_private_data=objectify_mask_paths(datamodule.test_private_data)
datamodule.test_private_mixed_data=objectify_mask_paths(datamodule.test_private_mixed_data)
datamodule.test_public_data=objectify_mask_paths(datamodule.test_public_data)
datamodule.train_data=objectify_mask_paths(datamodule.train_data)
datamodule.val_data=objectify_mask_paths(datamodule.val_data)

In [33]:
engine.fit(datamodule=datamodule, model=model)

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/antoine/manuf-anom-detector/.venv/lib/python3.12/site-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │ 24.9 M │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 24.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.9 M                                                                                               
Total estimated model params size (MB): 99.450                                                                     
Modules in train mode: 19                                                                                          
Modules in eval mode: 174                                                                                          
Total FLOPs: 0

/home/antoine/manuf-anom-detector/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/antoine/manuf-anom-detector/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:538: Found 174 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


DEBUG: train start
DEBUG: train batch start 0
DEBUG: train batch end 0
<class 'anomalib.models.image.patchcore.lightning_model.Patchcore'>
<class 'anomalib.models.image.patchcore.torch_model.PatchcoreModel'>
DEBUG store type: <class 'list'>
DEBUG: train batch start 1
DEBUG: train batch end 1
<class 'anomalib.models.image.patchcore.lightning_model.Patchcore'>
<class 'anomalib.models.image.patchcore.torch_model.PatchcoreModel'>
DEBUG store type: <class 'list'>
DEBUG: train batch start 2
DEBUG: train batch end 2
<class 'anomalib.models.image.patchcore.lightning_model.Patchcore'>
<class 'anomalib.models.image.patchcore.torch_model.PatchcoreModel'>
DEBUG store type: <class 'list'>
DEBUG: train batch start 3
DEBUG: train batch end 3
<class 'anomalib.models.image.patchcore.lightning_model.Patchcore'>
<class 'anomalib.models.image.patchcore.torch_model.PatchcoreModel'>
DEBUG store type: <class 'list'>
DEBUG: train batch start 4
DEBUG: train batch end 4
<class 'anomalib.models.image.patchcore.l

Selecting Coreset Indices.: 100%|██████████| 29797/29797 [01:32<00:00, 320.82it/s]


DEBUG: validation batch start 0


The validation set does not contain any anomalous images. As a result, the adaptive threshold will take the value of the highest anomaly score observed in the normal validation images, which may lead to poor predictions. For a more reliable adaptive threshold computation, please add some anomalous images to the validation set.
The validation set does not contain any anomalous images. As a result, the adaptive threshold will take the value of the highest anomaly score observed in the normal validation images, which may lead to poor predictions. For a more reliable adaptive threshold computation, please add some anomalous images to the validation set.


DEBUG: validation batch start 1


`Trainer.fit` stopped: `max_epochs=1` reached.


In [34]:
test_results = engine.test(datamodule=datamodule, model=model)

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/antoine/manuf-anom-detector/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.7882993817329407     │
│       image_F1Score       │    0.5416666865348816     │
│        pixel_AUROC        │    0.9192532300949097     │
│       pixel_F1Score       │    0.1270013153553009     │
└───────────────────────────┴───────────────────────────┘

Let's analyse these metrics, after a little introduction on metrics.
First, let's start by the beginning : TP, FP, FN, TN.

Positive : defect
Negative : normal

True Positive (TP) : Predicted defects, is a defect
False Positive (FP) : Predicted defects, is normal
True Negative (TN) : Predicted normal, is normal
False Negative (FN) : Predicted normal, is a defect

Now, a quick reflexion : in our context, do we want to favor a model that is a little bit conservative in predicting defects (ie a model predicting more FP than FN) or the other way around?

Considering we're looking at vials, I would favor more caution and prefer a model that predicts more FP than FN.

### Prediction match
In our example (vials), we're looking to predict if a vial image contains a defect or not.

### AUROC
First, at image level. As per Anomalib doc, AUROC stands for *Area Under the Receiver Operating Characteristic curve*, which is the area under the plot of the ROC graph (True positive rate as y, False Positive Rate as x). The bigger, the better. As I yet don't have any point of comparison, this will be my baseline.

Thus, my AUROC baseline is 0.78.

Auroc being threshold-independant, and does not tell me how many FN or FP I have at the selected threshold. It just tells me here that the model is reasonably competent at ranking defective images above normal ones, across thresholds. It's only part of the picture.

### F1 Score
F1 is defined as the harmonic mean of precision & recall (aka true positive rate, also sensitivity ; why so many names?) and here I get a rather disappointing score of 0.54. This doesn't tell me where I'm bad : is it precision ? (ie capacity of the model to predict true occurences only when they are indeed correct) or sensitivity (ie capacity of the model to not miss true values)